<!--
Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# Introduction to Custom Ops

You started this journey learning to extend PyTorch functionality by composing
existing functionality.

Then, with `torch.autograd.Function`, you learned the first technique to gain
increased control of PyTorch's core engines.

You learned some necessary background information on ops and compile, and now
you are ready to start creating your own custom ops that live alongside the
existing ATen ops.

### Custom ops via Python

Custom ops in Python are a well covered topic in the
[PyTorch docs](https://docs.pytorch.org/tutorials/advanced/python_custom_ops.html).

These custom ops have all the advantages of PyTorch's ATen ops: they compose
well with `torch.compile` to avoid a graph break, ultimately leading to better
performance.

### Custom ops via C++ HLO kernels

Stay tuned for a forthcoming guide to implementing custom ops via HLO kernels in
C++. That said, since `jax.lax` is a nearly 1:1 mapping to StableHLO ops, the
next section on kernels in JAX may give you the ability to generate the HLO ops
you want, in pure Python, without diving into C++ build details.

## Custom ops via JAX kernels (without Pallas)

JAX is a different framework from TorchTPU but it ultimately compiles down to
the same HLO as TorchTPU. In some cases, JAX may give you some additional
control over the HLO. For example, JAX provides the
[population_count op](https://docs.jax.dev/en/latest/_autosummary/jax.lax.population_count.html),
which lowers directly to the
[StableHLO popcnt op](https://openxla.org/stablehlo/spec#popcnt). This makes JAX
a solution to implement some custom ops.

As a toy problem, suppose you want to implement via JAX a custom op that
performs a quantized sum for the one-bit format developed in
**[Extending Modules via Composition](extending_modules_and_functions_via_composition.ipynb)**. You decide to take advantage of popcount as a "sum"
function for the one bit format. The sum of a tensor in this quantization format
can be inferred from the number of bits set *without conversion to a native
PyTorch floating point format*.

Each bit represents scale, and each non-bit represents -scale. So the sum is
simply

```
(number of bits set) * scale + (number of non-bits set) * (-scale)
= (number of bits set) * scale - (number of non-bits set) * scale
= ((number of bits set) - (number of non-bits set)) * scale
= ((number of bits set) - (total number of bits - number of bits set)) * scale
= (2 * (number of bits set) - (total number of bits)) * scale
```

Below is the sample implementation. Notice that:

*   `torch_tpu._internal.pallas.jax_op` wraps JAX inside a PyTorch custom op.
*   The caller of the PyTorch custom op has no visibility into the fact that the
    op is implemented in terms of JAX.

Using the logging techniques demonstrated previously, you can even trace down
the [`popcnt`](https://openxla.org/stablehlo/spec#popcnt) in the dump file
`module_0039.tt_jit_custom_kernel.before_optimizations.txt`.

```
%test_example__popcount_0xb92b15e71d7bc5ec.1 (packed_vector.1: u32[]) -> u32[] {
  %packed_vector.1 = u32[] parameter(0), sharding={replicated}, metadata={op_name="packed_vector"}
  ROOT %population_count.1 = u32[] popcnt(%packed_vector.1), metadata={op_name="jit(jax_popcount)/population_count" stack_frame_id=10}
}
```

`jax.lax.population_count` is not differentiable so you would only use it for
inference. For a more realistic `jax.lax` op you might want to expose, consider
`jax.lax.approx_max_k`.

For a canonical example of implementing a custom op using a JAX kernel,
including use of `jax.jvp`/`jax.grad` to automatically create a backward
function, see the unit test
`torch_tpu/tests/pallas/pallas_test.py`, specifically
`test_jax_dot_grad_for_backwards()`.

In [1]:
import os
import random
import sys
from typing import Callable, Final

# Inject flags and env var to libtpu. It sometimes needs flags, sometimes envvar.
LLO_DUMP_TO: Final[str] = "/tmp/llo_dump"
flags = [
    f"--xla_jf_dump_to={LLO_DUMP_TO}",
    "--xla_jf_dump_llo_text=true",
]
sys.argv.extend(flags)
os.environ["LIBTPU_INIT_ARGS"] = " ".join(flags)

XLA_DUMP_TO: Final[str] = "/tmp/xla_dump"
os.environ["XLA_FLAGS"] = (
    f"--xla_dump_to={XLA_DUMP_TO} --xla_dump_hlo_as_text"
)

from absl import app as absl_app  # pylint: disable=g-import-not-at-top
import torch  # pylint: disable=g-import-not-at-top
import torch_tpu._internal.pallas  # pylint: disable=g-import-not-at-top

# pylint: disable=protected-access
torch._logging.set_logs(aot_graphs=True)

I0000 00:00:1787340843.228343   83579 pjrt_api.cc:119] GetPjrtApi was found for TPU at /home/mateusz/miniforge3/envs/tpu-experiment/lib/python3.12/site-packages/libtpu/libtpu.so
I0000 00:00:1787340843.244981   83579 tpu_profiler_plugin.cc:791] PyTorch PrivateUse1ProfilerRegistry not found. Native TPU profiling will be disabled (this is expected on PyTorch < 2.12).
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
I0000 00:00:1787340843.266094   83579 device_rt.cc:130] PjRt runtime initialization deferred for tpu


Device type: tpu, Device index: default
Initializing TPU distributed runtime


In [2]:
# Copied from qat_linear.py
def pack32(values):
  """Packs a vector of 32 quantized values (signs) into a uint32"""
  assert values.size() == (32,)
  assert values.dtype == torch.float32
  assert torch.all((values == 1.0) | (values == -1.0))

  # Translate values from {-1, 1} to {0, 1}
  bits = (values == 1.0).to(torch.uint32)

  # Create all bit patterns from 0000, 0001, 0010, etc. on the same device.
  bit_patterns = 2 ** torch.arange(
      32, dtype=torch.uint32, device=values.device
  )

  # Summing the bit patterns is a bitwise_or reduction on a vector.
  return torch.sum(bits * bit_patterns, dim=0, dtype=torch.uint32)

In [3]:
def create_popcount_op() -> Callable:
  import jax

  def jax_popcount(packed_vector: jax.Array) -> jax.Array:
    """Returns the number of active bits in an Array."""
    return jax.lax.population_count(packed_vector)

  # jax_op takes name and fn as positional-only arguments.
  return torch_tpu._internal.pallas.jax_op(
      "test_example::popcount",
      jax_popcount,
  )

In [4]:
def dump_dir(directory):
  import os

  entries = os.listdir(directory) if os.path.exists(directory) else []
  print(f"\n=== Files in {directory} (Total: {len(entries)}) ===")

  # Only print filenames to keep execution fast
  for f in sorted(entries):
    print(f)

  print(
      "\n💡 Tip: All compilation files are written to disk. You can read any"
      " file interactively using: open(os.path.join(directory,"
      " filename)).read()"
  )

In [5]:
def main():
  # Verify device is TPU before running
  tpu_available = False
  try:
    _ = torch.randn(1, device="tpu")
    tpu_available = True
  except Exception:
    pass

  if not tpu_available:
    print(
        "TPU not available. Run this notebook on a TPU VM to execute"
        " quantized sum."
    )
    return 0

  popcount_op = create_popcount_op()

  def quantized_sum(packed_values, scale, total_bits):
    popcnt = popcount_op(packed_values)
    total_popcnt = popcnt.to(torch.int32).sum()
    return (2 * total_popcnt - total_bits) * scale

  num_ones = 7
  num_zeros = 25

  values = ([1.0] * num_ones) + ([-1.0] * num_zeros)
  random.shuffle(values)

  scale = 42.42
  expected_sum = sum(v * scale for v in values)

  # Create tensor directly on TPU and pack it
  data = torch.tensor(values, dtype=torch.float32, device="tpu")
  packed = pack32(data)

  # Call quantized_sum
  quantized_sum_val = quantized_sum(packed, scale, total_bits=32)

  dump_dir(XLA_DUMP_TO)
  dump_dir(LLO_DUMP_TO)

  print("\n=========================================")
  print("Quantized sum: ", quantized_sum_val.item())
  print("Expected sum: ", expected_sum)
  print("=========================================")
  print("Success.")
  return 0

main()

I0000 00:00:1787340843.582071   83579 pjrt_api.cc:168] The PJRT plugin has PJRT API version 0.114. The framework PJRT API version is 0.114.


I0000 00:00:1787340844.996590   83579 pjrt_c_api_client.cc:188] PjRtCApiClient created.
I0000 00:00:1787340845.013641   83815 compilation_cache_utils.cc:217] Tier-3 compilation cache root directory: 
I0000 00:00:1787340845.013672   83815 tier2_compilation_cache.cc:82] Tier-2 compilation cache is disabled for world size 1.
I0000 00:00:1787340845.013675   83815 tier3_compilation_cache.cc:50] Backup compilation for tier-3 cache read is disabled as tier-3 cache is disabled.
/tmp/ipykernel_83579/2781161803.py:5: UserWarning: the XLA_FLAGS environment variable is an experimental feature and may change or be removed without notice. (Triggered internally at ./torch_tpu/common/env_vars.h:211.)
  _ = torch.randn(1, device="tpu")



=== Files in /tmp/xla_dump (Total: 446) ===
module_0000.tt_jit_2117591624_L4C0_fwd_relu.cl_963683785
module_0000.tt_jit_2117591624_L4C0_fwd_relu.cl_963683785.after_codegen.txt
module_0000.tt_jit_2117591624_L4C0_fwd_relu.cl_963683785.after_optimizations-buffer-assignment-values.txt
module_0000.tt_jit_2117591624_L4C0_fwd_relu.cl_963683785.after_optimizations-buffer-assignment.txt
module_0000.tt_jit_2117591624_L4C0_fwd_relu.cl_963683785.after_optimizations-live-range.txt
module_0000.tt_jit_2117591624_L4C0_fwd_relu.cl_963683785.after_optimizations-memory-usage-report.txt
module_0000.tt_jit_2117591624_L4C0_fwd_relu.cl_963683785.after_optimizations.txt
module_0000.tt_jit_2117591624_L4C0_fwd_relu.cl_963683785.after_optimizations_after_buffer_assignment.txt
module_0000.tt_jit_2117591624_L4C0_fwd_relu.cl_963683785.after_optimizations_before_buffer_assignment.txt
module_0000.tt_jit_2117591624_L4C0_fwd_relu.cl_963683785.basic_compiler_metadata.txt
module_0000.tt_jit_2117591624_L4C0_fwd_relu.cl_9

0